# 1. Filter & Prepare Dataset for Sentence Transformer Fine-Tuning

This notebook downloads and filters a text-pair dataset (e.g., CodeSearchNet) for fine-tuning
a sentence transformer model. It filters pairs by token length to ensure they fit within
the model's max sequence length.

**Output:** A JSON file with filtered `(text_a, text_b)` pairs, saved to Google Drive.

This step is **optional** — you can skip it and bring your own dataset to the fine-tuning notebook.

## 1. Install Dependencies

In [ ]:
!pip install -q sentence-transformers datasets

## 2. Setup

In [ ]:
import json
import os
from pathlib import Path

import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

if IN_COLAB:
    drive.mount("/content/drive")

## 3. Configuration

Edit these to match your dataset and model.

In [ ]:
# === Dataset ===
HF_DATASET = "sentence-transformers/codesearchnet"  # HuggingFace dataset name
DATASET_SPLIT = "train"                              # which split to use
TEXT_A_FIELD = "comment"                             # first text field (e.g., query, docstring)
TEXT_B_FIELD = "code"                                # second text field (e.g., document, code)

# === Model (for tokenizer) ===
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_SEQ_LENGTH = 512  # filter out pairs where either text exceeds this token count

# === Output ===
OUTPUT_DIR = "/content/drive/MyDrive/sentence-transformer-finetune"
OUTPUT_FILENAME = "filtered_pairs.json"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

print(f"Dataset:        {HF_DATASET}")
print(f"Fields:         {TEXT_A_FIELD}, {TEXT_B_FIELD}")
print(f"Max tokens:     {MAX_SEQ_LENGTH}")
print(f"Output:         {output_path}")

## 4. Download & Filter

Downloads the dataset, tokenizes each pair, and keeps only pairs where both texts
fit within `MAX_SEQ_LENGTH` tokens.

In [ ]:
# Check if already filtered
if os.path.exists(output_path):
    print(f"Filtered dataset already exists: {output_path}")
    print("Delete the file to re-process.")
    with open(output_path, "r") as f:
        all_pairs = json.load(f)
    print(f"Loaded {len(all_pairs):,} pairs")
else:
    # Load tokenizer
    print(f"Loading tokenizer from: {BASE_MODEL}")
    temp_model = SentenceTransformer(BASE_MODEL)
    tokenizer = temp_model.tokenizer
    del temp_model

    # Download dataset
    print(f"\nDownloading {HF_DATASET}...")
    dataset = load_dataset(HF_DATASET, cache_dir=os.path.join(OUTPUT_DIR, "hf_cache"))

    data = dataset[DATASET_SPLIT]
    print(f"Processing {len(data):,} examples...")
    print(f"Filtering: both texts must be <= {MAX_SEQ_LENGTH} tokens\n")

    all_pairs = []
    processed = 0

    for item in data:
        text_a = item.get(TEXT_A_FIELD, "")
        text_b = item.get(TEXT_B_FIELD, "")

        if not text_a or not text_b:
            processed += 1
            continue

        try:
            tokens_a = len(tokenizer.encode(str(text_a), add_special_tokens=True))
            tokens_b = len(tokenizer.encode(str(text_b), add_special_tokens=True))

            if tokens_a <= MAX_SEQ_LENGTH and tokens_b <= MAX_SEQ_LENGTH:
                all_pairs.append({"text_a": str(text_a), "text_b": str(text_b)})
        except Exception:
            pass

        processed += 1
        if processed % 50000 == 0:
            print(f"  Processed {processed:,}, kept {len(all_pairs):,} pairs...")

    # Save
    print(f"\nSaving {len(all_pairs):,} filtered pairs to {output_path}...")
    with open(output_path, "w") as f:
        json.dump(all_pairs, f)

    print(f"\nDone! {processed:,} processed, {len(all_pairs):,} kept ({100 * len(all_pairs) / max(1, processed):.1f}%)")

## 5. Quick Stats

In [ ]:
print(f"Total pairs: {len(all_pairs):,}")
print(f"\nSample pair:")
if all_pairs:
    sample = all_pairs[0]
    print(f"  text_a: {sample['text_a'][:150]}...")
    print(f"  text_b: {sample['text_b'][:150]}...")

print(f"\nFile saved to: {output_path}")
print(f"Use this path in the fine-tuning notebook's DATA_PATH config.")